In [2]:
import os
import subprocess

def find_starlink_setup(base_paths, filename="profile"):
    """Recursively search for a file named `filename` starting from `base_paths`."""
    for base_path in base_paths:
        for root, dirs, files in os.walk(base_path):
            if filename in files:
                return os.path.join(root, filename)
    return None

def run_command(command, cwd=None):
    """Run a command and print its output."""
    print(f"Running command: {command}")
    result = subprocess.run(command, shell=True, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        print(f"Error running command: {result.stderr}")
    else:
        print(result.stdout)

def process_fits_files(folder_path, starlink_setup_path, gaussian_fwhm=2.7, smoothed_suffix="_23arcsec"):
    # Source the Starlink environment
    source_command = f"source {starlink_setup_path}"
    
    # List all .fits files in the folder
    for filename in os.listdir(folder_path):
        if filename.endswith("rms.fits"):
            fits_file = os.path.join(folder_path, filename)
            base_name = os.path.splitext(filename)[0]

            # Convert FITS to NDF
            ndf_file = os.path.join(folder_path, f"{base_name}.sdf")
            run_command(f"  /usr/local/star-2021A-REV1/bin/convert/fits2ndf {fits_file} {ndf_file}")

            # Apply Gaussian smoothing to the FITS file
            smoothed_ndf = os.path.join(folder_path, f"{base_name}{smoothed_suffix}.sdf")
            run_command(f" /usr/local/star-2021A-REV1/bin/kappa/gausmooth {ndf_file} FWHM={gaussian_fwhm} OUT={smoothed_ndf}")

            # Convert the smoothed NDF back to FITS
            output_fits = os.path.join(folder_path, f"{base_name}{smoothed_suffix}.fits")
            run_command(f" /usr/local/star-2021A-REV1/bin/convert/ndf2fits {smoothed_ndf} {output_fits}")

if __name__ == "__main__":
    # Define the folder where the FITS files are located
    FOLDER_PATH = "/Users/kyriakos.trakakis/Downloads/CHIMPS_COHRS/C18O"

    # Define common base directories to search for Starlink setup
    BASE_DIRS = [
        "/usr/local",
        "/usr",
        "/opt",
        "/home",
    ]
    
    # Find the Starlink setup script
    starlink_setup_path = find_starlink_setup(BASE_DIRS)
    starlink_setup_path=  '/usr/local/star-2023A-REV1/bin/convert/fits2ndf'
    if not starlink_setup_path:
        print("Starlink setup script not found.")
        exit(1)

    print(f"Found Starlink setup script: {starlink_setup_path}")

Found Starlink setup script: /usr/local/star-2023A-REV1/bin/convert/fits2ndf


In [3]:
starlink_setup_path

'/usr/local/star-2023A-REV1/bin/convert/fits2ndf'

In [4]:
## Smooth a set of files to 23 arcsec. I will try to do this all in one go,
## so turn to sdf, smooth, turn back to fits.

FOLDER_PATH = "/Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/"
#FOLDER_PATH = "/Users/kyriakos.trakakis/Downloads/FUGIN_data"
folder_path=FOLDER_PATH

for filename in os.listdir(FOLDER_PATH):
        if filename.endswith(".fits"):
            # Get the filename
            fits_file = os.path.join(FOLDER_PATH, filename)
            base_name = os.path.splitext(filename)[0]

            # Turn to ndf
            ndf_file = os.path.join(folder_path, f"{base_name}.sdf")
            run_command(f"  /usr/local/star-2023A/bin/convert/fits2ndf {fits_file} {ndf_file}")

            # Smooth from 6 arcsec (CHIMPS/COHRS) to 23, fwhm = 3.83
            # Smooth from 8.5 arcsec (FGN) to 23, fwhm = 2.7
            # Smooth Mopra from 30 arcsec to 40, fwhm = 1.333
            smoothed_suffix ="_23arcsec"
            #smoothed_suffix ="_1beam"
            gaussian_fwhm= 2.7
            smoothed_ndf = os.path.join(folder_path, f"{base_name}{smoothed_suffix}.sdf")
            run_command(f" /usr/local/star-2023A/bin/kappa/gausmooth {ndf_file} FWHM={gaussian_fwhm} OUT={smoothed_ndf}")

            # ndf back to fits
            output_fits = os.path.join(folder_path, f"{base_name}{smoothed_suffix}.fits")
            run_command(f" /usr/local/star-2023A/bin/convert/ndf2fits {smoothed_ndf} {output_fits}")

            # Delete ndfs to save space
           # os.system('rm -rf '+smoothed_ndf)
           # os.system('rm -rf '+ndf_file)


Running command:   /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/FGN_219-05_C18O.fits /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/FGN_219-05_C18O.sdf
1 file selected.

Running command:  /usr/local/star-2023A/bin/kappa/gausmooth /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/FGN_219-05_C18O.sdf FWHM=2.7 OUT=/Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/FGN_219-05_C18O_23arcsec.sdf

Running command:  /usr/local/star-2023A/bin/convert/ndf2fits /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/FGN_219-05_C18O_23arcsec.sdf /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/FGN_219-05_C18O_23arcsec.fits
1 NDF selected.
The 1st NDF
(/Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/C18O/FGN_219-05_C18O_
23arcsec) does not have the selected QUALITY and VARIANCE components.
The output FITS file will be produced for the remaining array components that
were spe

In [3]:
folder_path = "/Users/kyriakos.trakakis/Downloads/FUGIN_data/"
for filename in os.listdir(folder_path):
    if filename.endswith("rms.fits"):
        fits_file = os.path.join(folder_path, filename)
        base_name = os.path.splitext(filename)[0]
        print(fits_file)

/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_13CO_021_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_037_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_041_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_13CO_041_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_015_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_13CO_037_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_021_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_13CO_047_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_13CO_031_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_013_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_027_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_019_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_13CO_027_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FGN_12CO_031_rms.fits
/Users/kyriakos.trakakis/Downloads/FUGIN_data/FG

In [5]:
for filename in os.listdir(folder_path):
        if filename.startswith("FGN"):
            # Get the filename
            fits_file = os.path.join(FOLDER_PATH, filename)
            base_name = os.path.splitext(filename)[0]
            print(base_name)

FGN_13CO_021_rms
FGN_12CO_037_rms
FGN_12CO_041_rms
FGN_13CO_041_rms
FGN_12CO_015_rms
FGN_13CO_037_rms
FGN_12CO_021_rms
FGN_13CO_047_rms
FGN_13CO_031_rms
FGN_12CO_013_rms
FGN_12CO_027_rms
FGN_12CO_019_rms
FGN_13CO_027_rms
FGN_12CO_031_rms
FGN_12CO_047_rms
FGN_13CO_049_rms
FGN_12CO_023_rms
FGN_12CO_029_rms
FGN_13CO_043_rms
FGN_12CO_017_rms
FGN_13CO_035_rms
FGN_12CO_035_rms
FGN_13CO_029_rms
FGN_12CO_043_rms
FGN_12CO_049_rms
FGN_13CO_023_rms
FGN_12CO_033_rms
FGN_12CO_045_rms
FGN_12CO_039_rms
FGN_13CO_025_rms
FGN_12CO_025_rms
FGN_13CO_039_rms
FGN_13CO_045_rms
FGN_13CO_033_rms
FGN_12CO_011_rms


In [6]:
os.getcwd()

'/Users/kyriakos.trakakis/Downloads/FUGIN_data'

In [3]:
FOLDER_PATH = "/Users/kyriakos.trakakis/Downloads/FUGIN_data"
for filename in os.listdir(FOLDER_PATH):
        if filename.startswith("FGN"):
            # Get the filename
            fits_file = os.path.join(FOLDER_PATH, filename)
            base_name = os.path.splitext(filename)[0]

            # Turn to ndf
            ndf_file = os.path.join(FOLDER_PATH, f"{base_name}.sdf")
            run_command(f"  /usr/local/star-2023A/bin/convert/fits2ndf {fits_file} {ndf_file}")

Running command:   /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_226-05_12CO_rms.fits /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_226-05_12CO_rms.sdf
1 file selected.

Running command:   /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_218-05_12CO_rms.fits /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_218-05_12CO_rms.sdf
1 file selected.

Running command:   /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_217+05_12CO.fits /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_217+05_12CO.sdf
1 file selected.

Running command:   /usr/local/star-2023A/bin/convert/fits2ndf /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_202+05_12CO.fits /Users/kyriakos.trakakis/Downloads/Mopra_FGN_project/FGN/FGN_202+05_12CO.sdf
1 file selected.

Running command:   /usr/local/star-2023A/bin